# 1 Setting Up the Environment

Follow the steps presented at the `README.md` file to install both PyTorch and the Minkowski Engine. And check below if the installation was successful.

In [1]:
import MinkowskiEngine as ME
print(f'MinkowskiEngine version: {ME.__version__}')
import torch
print(f'PyTorch version: {torch.__version__}')

/home/corteletti/miniconda3/envs/fcgf/lib/python3.9/site-packages/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(


MinkowskiEngine version: 0.5.4
PyTorch version: 2.5.1


/home/corteletti/miniconda3/envs/fcgf/lib/python3.9/site-packages/MinkowskiEngine/__init__.py:221: UserWarning: The MinkowskiEngine was compiled with CPU_ONLY flag. If you want to compile with CUDA support, make sure `torch.cuda.is_available()` is True when you install MinkowskiEngine.
  warnings.warn(


Then, we can also check if all of FCGF's extra requirements listed at the `FCGF/requirements.txt` file were properly installed.

In [2]:
%pip install -r ../source/FCGF/requirements.txt

Note: you may need to restart the kernel to use updated packages.


After that, we just need to import everything we are going to use.

In [ ]:
import os
import io
import sys
import copy
import math
import time
import shutil
import subprocess
import numpy as np
import pandas as pd
import open3d as o3d
from functools import wraps
from urllib.request import urlretrieve
from collections import defaultdict
from datetime import datetime
from zoneinfo import ZoneInfo
from typing import Literal
from contextlib import redirect_stdout

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


And set Python to include `source/FCGF` in its search path. Otherwise, since this notebooks is in a different folder, we would not be able to import functions and other things from the FCGF folder.

In [4]:
# Get the absolute path of the source directory
sys.path.append(os.path.abspath("../source/FCGF"))

---

# 2 Input Data

Since we will not retrain the model but use just the pre-trained weights, we can download only the test split. If the train split is ever needed, you can download it too by uncommenting the last block.

In [5]:
test_path = '../data/FCGF/threedmatch_test'
if not os.path.exists(test_path):
  print(f'Downloading data at {test_path}\n\n=================================================================\n')
  subprocess.run(["bash", "../source/FCGF/scripts/download_3dmatch_test.sh", test_path], check=True)
else:
    print(f'The data is already available at {test_path}')


# train_path = '../data/FCGF/threedmatch_train/'
# if not os.path.exists(train_path):
#   print(f'Downloading data at {train_path}\n=================================================================')
#   !bash ../source/FCGF/scripts/download_datasets.sh {train_path}
# else:
#     print(f'The data is already available at {train_path}')

The data is already available at ../data/FCGF/threedmatch_test


Besides that, we also need to download the pre-trained model so we can use it to perform the tests.

In [6]:
# Check if the weight folder has already been created, otherwise creates it
fcgf_weights_folder = '../weigths/FCGF'
if not os.path.exists(fcgf_weights_folder):
    os.makedirs(fcgf_weights_folder)

# Check if the selected pre-trained weights were already downloaded, otherwise download them
fcgf_weight = 'ResUNetBN2C-16feat-3conv.pth'
fcgf_weight_path = os.path.join(fcgf_weights_folder, fcgf_weight)
if not os.path.isfile(fcgf_weight_path):
  print(f'Downloading weight at {fcgf_weight_path}...')
  urlretrieve("https://node1.chrischoy.org/data/publications/fcgf/2019-09-18_14-15-59.pth",
              fcgf_weight_path)
else:
    print(f'Selected weights already available at {fcgf_weight_path}')

Selected weights already available at ../weigths/FCGF/ResUNetBN2C-16feat-3conv.pth


---

# 3 Local Refinement

The current registration pipeline defined in `source/FCGF/scripts/benchmark_3dmatch.py` uses only a global registration stage through feature-based RANSAC with the features extracted by FCGF. To improve the results, we introduce a local refinement stage using ICP, as done in the geometric-only pipeline.

## 3.1 Retrieving Transformations from Logs

When running the `benchmark_3dmatch.py` script, it executes RANSAC for all pairs. However, unlike the geometric-only pipeline, it does not return a results table. Instead, it saves the obtained transformations as log files in the output folder.

Since we first run RANSAC for all pairs and only then we can apply ICP, we need a function to retrieve the transformations computed by RANSAC so we can use them as input for ICP. Thus, below we define a function that extracts the transformation from any specified log path.

In [7]:
def get_transformation_from_content(content: list[str], line_idx: int) -> np.ndarray:
    """Extract the 4×4 transformation matrix from pre-loaded log lines.

    The log is organized in 5-line blocks per pair:
      1) Header: "<tgt_ID> <src_ID> <nFrags>"
      2–5) Four rows of the 4×4 matrix.

    Args:
        content (list[str]): All lines of the log file, as returned by `f.readlines()`.
        line_idx (int): Index of the header line in `content`.

    Returns:
        np.ndarray: A (4,4) float array containing the transformation matrix.
    """

    # List to accumulate the next four rows of the matrix as lists of floats
    transformation = []

    # Iterate 4 times (since the matrix is 4x4 -> 4 rows
    for i in  range(4):

        # Update the current row of the matrix
        line_idx += 1
        
        # Extract, cleans newline/tab characters and split the line into tokens
        line = content[line_idx].strip().split()
        
        # Convert each element to float and append as one row
        transformation.append([float(value) for value in line])

    return np.array(transformation)

## 3.2 Adding an ICP Stage

Now, we need to swipe through all the logs, extract the RANSAC transformation (with the help of the previous function), apply the ICP stage, and save the ICP result to a new log file in the output directory.

To do this, we’ll implement the same ICP function that was used in the following notebooks:
- `3-ICP_Pipeline_Datasets.ipynb`  
- `2-GlobalRegistration_demo.ipynb`  
- `1-ICP_demo.ipynb`

Besides that, we also need to implement the `timer` decorator which will be used to time the ICP stage, as it was done for the `3-ICP_Pipeline_Datasets.ipynb` notebook. Notice that the other stages (preprocessing and RANSAC) will be handled by the `FCGF/scripts/benchmark_3dmatch.py` and `FCGF/scripts/benchmark_util.py` scripts, so these timers must be added there. For simplicity, a copy of the timer decorator definition was made in the `FCGF/scripts/`directory, to facilitate its use across both scripts. [It would be harder to try to import it from here to the FCGF directtory]

In [8]:
# A global dict that accumulates total elapsed time for each named stage
# Keys are stage names (strings), values are floats (seconds)
total_stage_times = defaultdict(float)

def timer(stage_name):
    """
    Decorator factory: creates a decorator that wraps a function,
    measures its execution time, and adds that time to
    total_stage_times[stage_name]
    OBS: uses a decorator factory to be able to do @time('name of the stage')
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = func(*args, **kwargs)
            elapsed = time.perf_counter() - start
            total_stage_times[stage_name] += elapsed            
            return result
        return wrapper
    return decorator

In [9]:
@timer('icp')
def execute_ICPrefinement(source, target, inlier_th, trans_init, voxel_size):
    """Executes the local ICP refinement for an initial transformation.

    Args:
        source (open3d.geometry.PointCloud): Source cloud
        target (open3d.geometry.PointCloud): Target cloud
        inlier_th (float): Threshold distance for a correspondence pair to be considered valid
        trans_init (numpy.ndarray): Initial transformation matrix
        voxel_size (float): Resulting size of voxels after downsampling

    Returns:
        open3d.pipelines.registration.RegistrationResult: Class that contains the registration results
    """

    #target normals estimation
    radius_normal = 2*voxel_size
    target.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))

    #performs the point-to-plane ICP
    ICP_registration = o3d.pipelines.registration.registration_icp(source, target, inlier_th, trans_init,
                                                                   o3d.pipelines.registration.TransformationEstimationPointToPlane())
    return ICP_registration

Then, as mentioned, we also need to save these new refined results to log files, just as we did for the RANSAC results. To do this, we define a function that writes a log file for a given pair.

In [10]:
def write_refined_log(icp_log_path, tgt_ID, src_ID, nFrags, transformation):
    """Append a refined registration result (ICP output) to the scene’s log file.

    Parameters
    ----------
    icp_log_path : str
        Path for the refined log file that will store the icp results.
    tgt_ID : int
        Target fragment ID.
    src_ID : int
        Source fragment ID.
    nFrags : int
        Number of fragments used in the initial guess.
    transformation : np.ndarray, shape (4,4)
        4×4 homogeneous transformation matrix from ICP refinement.
    """
        
    # Use mode 'a' to append new entries rather than overwrite existing ones
    with open(icp_log_path, 'a') as f:

        # First line: IDs and fragment count
        f.write(f'{tgt_ID} {src_ID} {nFrags}\n')

        # Next 4 lines: the rows of the 4×4 transformation matrix
        for row in transformation:
            # Format each value to 12 decimal places with a space between them
            line = " ".join(f"{val:.12f}" for val in row)
            f.write(f'{line}\n')

Then, we can implement the global sweeping behavior through the following function. It loops through all the logs in the specified output folder and processes each of the present pairs. 

For each pair, it retrieves the RANSAC transformation along with its evaluation metrics (fitness and inlier RMSE), applies ICP refinement, and saves the refined results to the logs folder. 

This function also generates a results table containing all information from both stages (RANSAC and ICP). Notice that in the previous versions of this notebook, when we only had RANSAC on the FCGF pipeline, this was done by a separate function `get_results_from_folder`. However, since we must loop through the logs to compute the ICP refinement, we can take advantage to simultaneously retrieve all performance information from the logs, avoiding to have to loop through the logs a second time. Therefore, the behavior of that function was included here and we can then discard it.

In [11]:
def ICP_stage(output_folder, test_path, inlier_th, voxel_size):
    """
    Run a full ICP pipeline over all initial guesses logs and collect results.

    For each scene, reads the initial-guess logs, performs:
      1. RANSAC evaluation on full-resolution clouds.
      2. ICP refinement starting from the RANSAC transformation.
      3. Logs the refined transformation.
      4. Appends metrics to a list of results which is returned as a DataFrame.

    Parameters
    ----------
    output_folder : str
        Base output folder containing 'registration/initial_guesses_logs'.
    test_path : str
        Root folder where point-cloud scene subdirectories reside.
    inlier_th : float
        Distance threshold for inlier determination in evaluation.
    voxel_size : float
        Downsampling voxel size used in ICP refinement.

    Returns
    -------
    pd.DataFrame
        Table of metrics and transformations for each source–target pair.
    """    
    
    results = []

    # Directory containing one log per scene of initial FCGF guesses (RANSAC)
    ransac_logs_folder = f'{output_folder}/registration/initial_guesses_logs'

    # Define the refined logs directory
    icp_logs_folder = f"{output_folder}/registration/logs"

    # If the folder already exists, deletes it to remove the old entries
    # otherwise it would append the entries of the new run on top of the old ones
    if os.path.exists(icp_logs_folder):    
        shutil.rmtree(icp_logs_folder)     
    
    # Creates a new folder
    os.makedirs(icp_logs_folder)

    # Iterate over each log file in the folder
    for log in os.listdir(ransac_logs_folder):
        
        # Extract scene name (before the '_FCGF' suffix in filename)
        scene = log.split('_FCGF')[0]            
        print(f'Set: {scene}')

        # Define the ransac log path of a given scene
        ransac_log_path = f'{ransac_logs_folder}/{log}'

        # Define the current scene's refined log path
        icp_log_path = f'{icp_logs_folder}/{scene}_FCGF.log'

        with open(ransac_log_path, 'r') as f:

            # Read all lines once
            content = f.readlines()
            n = len(content)

            # Each entry consists of 5 lines: one header + 4 rows of matrix
            # so the step is set to 5 to read only the headers
            for i in range(0, n, 5):

                # Access the line, clean and split it
                line = content[i]
                line = line.strip().split()

                # Retrieve the pair information
                tgt_ID = int(line[0])
                src_ID = int(line[1])
                nFrags = int(line[2])

                # Load the related point clouds
                source = o3d.io.read_point_cloud(os.path.join(test_path, scene, 'cloud_bin_%s.ply' %src_ID))
                target = o3d.io.read_point_cloud(os.path.join(test_path, scene, 'cloud_bin_%s.ply' %tgt_ID))
                
                # Retrieves the RANSAC transformation of the pair
                ransac_tran = get_transformation_from_content(content, i)
                
                # Evaluate RANSAC on the full clouds
                # (if we use ransac_reg.fitness or ransac_reg.inlier_rmse, these
                #  results would be computed on the downsampled clouds used by RANSAC)
                ransac_eval = o3d.pipelines.registration.evaluate_registration(source, target, inlier_th, ransac_tran)

                print(f'\tMatching {tgt_ID:03d} {src_ID:03d}')

                # Set Open3D's verbosity level to Debug to capture detailed iteration information
                o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Debug)

                # Refine the RANSAC guess with ICP
                icp_reg = execute_ICPrefinement(source, target, inlier_th, ransac_tran, voxel_size)

                # Returns Open3D's verbosity level to default mode
                o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)

                # Append the refined transformation to the log
                write_refined_log(icp_log_path, tgt_ID, src_ID, nFrags, icp_reg.transformation)

                # Collect all information for the results table
                new_result = {'Scene': scene,
                              'Target': tgt_ID,
                              'Source': src_ID,
                              'RANSAC: Fitness': ransac_eval.fitness,
                              'ICP: Fitness': icp_reg.fitness,
                              'RANSAC: Inlier RMSE': ransac_eval.inlier_rmse,
                              'ICP: Inlier RMSE': icp_reg.inlier_rmse,
                              'Initial Guess': ransac_tran, #wrapper on matrix -> check notebok 3-ICP_Pipeline_Datasets for details
                              'Transformation': icp_reg.transformation
                }
                results.append(new_result)

                print('\tDone')

        # Print the refined log path for that scene
        print(f'\tLogging at:" {icp_log_path}')

    # Convert the collected results to a pandas DataFrame
    results_table = pd.DataFrame(results)
    return results_table

---

# 4 Iteration Counter

Similar to the approach used in the notebook `3-ICP_Pipeline_Datasets.ipynb`, we need to add iteration counters for both the RANSAC and ICP stages. For a detailed explanation on the logic behind the counter implementation, please refer to the aforementioned notebook. Here, we only highlight the adaptations from the previous version to match the case of this notebook:
- Since we are using the author's scrip to run FCGF (with our adaptations to fit our casee), we are first running RANSAC for all pairs (using the script) and only then do we run ICP for all pairs as well (using the function we just defined in section 3.2). Thus, we need to define two separate cases for the iteration counters, one for each stage, which must be specified when calling the function.
- For the RANSAC stange, since it is executed by the script, its console outputs are always preceded by data and time. So, we must adjust the position index of the keywords used to obtain relevant lines (e.g., lines containing `'Set:'` to indicate a scene, or `'Matching'` to indicate a specific target-source pair).
- For ICP, instead of checking for `'Overlap'` to mark the end of the alignment for a specific pair (as done in RANSAC), we must now look for `'Done'`, which is now the output indicating completion of an alignment.



In [12]:
def get_iterations(captured_output, stage: Literal['RANSAC', 'ICP']):
    """
    Parses the captured output obtained in bedug mode to extract
    the iteration counts for each alignment.

    Parameters
    ----------
    captured_output : str
        Vriable storing the captured output
    stage : str
        Specifies from which stage the iterations should be extracted. Must be either 'RANSAC' or 'ICP'.

    Returns
    -------
    pd.DataFrame
        Table with the iteration count for the given stage.
    """   

    if stage not in ('RANSAC', 'ICP'):
        raise ValueError(f"Invalid stage '{stage}'. Must be 'RANSAC' or 'ICP'.")
        print(f"Using stage: {stage}")

    # List to store iteration data for each alignment
    iterations = []

    # Clean the captured output: remove tabs, extra spaces, and split it into individual lines.
    captured_output = captured_output.replace('[Open3D DEBUG]', '').replace('\t', '').strip().split('\n')

    # Loops through the output lines
    for line in captured_output:

        # Split the line into words
        line = line.split()
        if not line:  # Skip empty lines
            continue

        # Check the case (and length to avoid index errors)
        if stage == 'RANSAC' and len(line) >= 3:

            # Get current scene
            if line[2] == 'Set:':           # checks if it starts with 'Set:'
                cur_scene = line[3]   

            # Get currrent matching pair
            elif line[2] == 'Matching':     # Checks if it starts with Matching
                target_ID = int(line[3])    # Obtain the target ID from the second field
                source_ID = int(line[4])    # Obtain the target ID from the third field
                ICP_iterations = 0   
                
            # Get ransac iterations
            elif line[0] == 'RANSAC':
                RANSAC_iterations = int(line[3])
            
            # When the line starts with 'Overlap', it signals the end of the current alignment
            # If the alignment was achieved (30% overlap), store the results
            elif line[2] == 'Overlap' and float(line[4]) > 0.3: 
                
                # Stores the iterations of the current pair
                # Appending dictionaries is to a list is a straightforward way of build DataFrames later
                # Each dictionary in the list is a row of the table and the columns are the dictionaries keys
                iterations.append({
                    'Scene': cur_scene,
                    'Target': target_ID,
                    'Source': source_ID,
                    'RANSAC Iterations': RANSAC_iterations
                })

        # When dealing with the refinement stage
        elif stage == 'ICP':
                
            # Get current scene
            if line[0] == 'Set:':           # checks if it starts with 'Set:'
                cur_scene = line[1]         # extracts the scene name from the following text in the line

            # Get currrent matching pair
            elif line[0] == 'Matching':     # Checks if it starts with Matching
                target_ID = int(line[1])    # Obtain the target ID from the second field
                source_ID = int(line[2])    # Obtain the target ID from the third field
                ICP_iterations = 0          # Resets the ICP iterations counter for the next pair

            # If the line indicates an ICP iteration, increment the ICP iterations counter
            elif line[0] == 'ICP':
                ICP_iterations += 1

            # When the alignment is completed, store it in the list
            elif line[0] == 'Done':
                iterations.append({
                        'Scene': cur_scene,
                        'Target': target_ID,
                        'Source': source_ID,
                        'ICP Iterations': ICP_iterations
                    })
                # print(f'{cur_scene} {target_ID} {source_ID}')
                # print('--------------------------------------')

    iterations_df = pd.DataFrame(iterations)
    return iterations_df

Then we can define a function responsible merging the iteration information with our main results DataFrame. We perform a left merge on the 'Scene', 'Target', and 'Source' columns so that every row in the results table is augmented with the corresponding iteration counts. We then reorder the columns for better visualization.

In [13]:
def augment_results(results, ransac_captured_output, icp_captured_output):
    """Augments the main results DataFrame with RANSAC and ICP iteration counts.

    This function merges the provided `results` DataFrame with iteration count DataFrames
    extracted from captured console outputs of the RANSAC and ICP stages. The merging is 
    done as a left join on the 'Scene', 'Target', and 'Source' columns. The resulting 
    DataFrame is then reordered to place the iteration columns in more intuitive positions.

    Args:
        results (pd.DataFrame): The main DataFrame containing alignment results.
        ransac_captured_output (str): Captured console output from the RANSAC stage.
        icp_captured_output (str): Captured console output from the ICP stage.

    Returns:
        pd.DataFrame: The augmented DataFrame containing iteration counts for RANSAC and ICP.
    """
    
    # Extract iteration counts from the captured outputs
    ransac_iter_df = get_iterations(ransac_captured_output, stage='RANSAC')
    icp_iter_df = get_iterations(icp_captured_output, stage='ICP')
        
    # Merge the iteration counts into the results DataFrame
    augmented_results = results.merge(ransac_iter_df, 'left', on=['Scene', 'Target', 'Source'])
    augmented_results = augmented_results.merge(icp_iter_df, 'left', on=['Scene', 'Target', 'Source'])

    # Reoders the columns
    col_names = list(augmented_results.columns.values)  # get labels
    col_names.insert(3, col_names[-2])                  # insert 2nd to last item (RANSAC Iterations) at index 3
    col_names.pop(-2)                                   # removes 2nd to last item
    col_names.insert(4, col_names[-1])                  # insert last item (ICP Iterations) at index 4
    col_names.pop()                                     # removes last item
    augmented_results = augmented_results[col_names]    # reorder df with the same order of the col_names list
    
    return augmented_results

As mentioned in the previous notebook, another obstacle is redirecting the output from the console to a variable, which must be inputed in the functions we just defined. Therefore, once again we use a `Tee` object. For further details on this, please refer to notebook `3-ICP_Pipeline_Datasets.ipynb`.

In [14]:
class Tee(io.StringIO):
    """
    A 'tee' that writes output to multiple streams:
    - the console (sys.__stdout__)
    - an internal buffer (which you can retrieve later)
    """
    def write(self, text):
        # write to console
        sys.__stdout__.write(text)
        # Also write to this StringIO's buffer (use super() since it is the parent class of this one)
        super().write(text)

    def flush(self):
        # Ensure both this buffer and console are flushed
        sys.__stdout__.flush()
        super().flush()

---

# 5 Defining the Complete Pipeline

The final step is to implement a function that structures the flow of the entire pipeline. The steps are as follows:
- Optional (if a subset was selected): check if the selected subset value is valid **(*)**
- Run RANSAC on all target-source pairs and capture the output.
- Run ICP on all pairs and capture the output.
- Extract iteration counts from the captured outputs and merge everything into a final results table.

<br>

**(*) NOTE**: The verification step is necessary when we are dealing with a subset test case because not all values of subsets (i.e pairs) would be feasible, since there are some amount of pairs that cannot be achieved by combining any integer amount of point clouds. To explain this, knowing that the pairs are constructed from the clouds, we need to combine the number of clouds N into P pairs using the combinatory equation:

$$ P = C(N,2) = \frac{N!}{2!(N-2)!} = \frac{N(N-1)}{2} $$

Solving for N:

$$ N^2 -N -2P = 0 $$

Using the quadratic formula:

$$ \Delta = (-1)^2 -4 \cdot 1 (-2P) = 1 + 8P $$
$$ N = \frac{(-1)^2 \pm \sqrt{\Delta}}{-2 \cdot (-1)} = \frac{1 \pm \sqrt{\Delta}}{2} $$

Since $ P \geq 0 $, then N must also be positive (we cannot have a negative number of point clouds) and we can discard the negative branch

$$ N = \frac{1 + \sqrt{\Delta}}{2} $$

Therefore, in order for N to be integer we must check that:
- $\sqrt{\Delta}$ is a integer, which means that $ \Delta = 1+8P$ must be a perfect square
- $ 1+\sqrt{\Delta} $ is divisible by two

> There are easier ways of implementing this behavior by computing N from the given P (subset) and checking whether the obtaineeed N is an integer. However, due to computer precision, it can sometimes results in unexpected behaviors (for instance, instead of N being 3, it could receive 3.0000000004, causing the integer check to fail). To work around this, we could use the `round()` function to discard this small floating point variances. But in any case, the approach presented here is the most robust.

In [26]:
def check_feasible_subset(subset: int) -> None:
    """
     Checks if `subset` is a valid number of pairs to be considered.
     Raises ValueError if it is not. Otherwise returns None and the script continues.
     """
   
    # Check for negative values
    if subset < 0:
        raise ValueError("subset must be non-negative")

    # Compute delta
    delta = 1 + 8*subset

    # Perform the checks described previously
    root = math.isqrt(delta)                        # .isqrt() returns the nearest smaller integer of the sqrt 
    if root*root == delta and (1+root) % 2 == 0:
        return
    
    # If subset is not valid, raise error and display nearest feasible value
    else:
        # Compute the exact float N obtained from the quadratic formula
        N = (1+root)/2
        
        # By forcing it to int, we are truncating the decimals and rounding it down
        N_down = int(N)

        #Therefore, the upper bound is simply the lower plus 1
        N_up = N_down + 1  

        # Compute the bounds of feasible subset values (use // to once again avoid FP errors)
        P_lower = N_down*(N_down-1)//2
        P_upper= N_up*(N_up-1)//2

        # raise error 
        raise ValueError(
            f"subset={subset} is not a valid triangular number.\n"
            f"--> Nearest smaller feasible subset: {P_lower}\n"
            f"--> Nearest larger  feasible subset: {P_upper}"
        )

def run_benchmark(test_path, feature_path, voxel_size, inlier_th, subset, model, stdout=None):
    """
    Invoke the 3DMatch benchmark script via subprocess, streaming its stdout
    to both the real console and an optional buffer.
    """
    # Build the command-line invocation
    args = [
        sys.executable,                                # use same Python interpreter
        "../source/FCGF/scripts/benchmark_3dmatch.py", # path to the benchmark driver
        "--source", test_path,                         # input scenes directory
        "--target", feature_path,                      # features output directory
        "--voxel_size", str(voxel_size),               # downsampling parameter
        "--inlier_th", str(inlier_th),                 # inlier threshold
        "--model", model,                              # FCGF model weigths
        "--extract_features",                          # first stage: extract features
        "--evaluate_feature_match_recall",             # compute match recall
        "--evaluate_registration",                     # run geometric registration
    ]
    
    # If running a subset of pairs, add the flag
    if subset:
        args += ["--subset", str(subset)]
    
    # Launch the child process with its stdout piped back to us
    proc = subprocess.Popen(
        args,
        stdout=subprocess.PIPE,  # create a real OS pipe for stdout
        stderr=subprocess.DEVNULL,  # ignore everything on stderr
        text=True,               # decode output as text (not bytes)
        bufsize=1                # line-buffered mode for timely output
    )


    # Read each line from the child’s stdout as it arrives
    for line in proc.stdout:
        # Echo to the real console immediately
        sys.__stdout__.write(line)
        # Also store into our provided buffer, if any
        if stdout is not None:
            stdout.write(line)

    # Close our reading end, wait for the process to exit
    proc.stdout.close()
    ret = proc.wait()

    # If the child exited with a non-zero code, raise an exception
    if ret != 0:
        raise subprocess.CalledProcessError(ret, args)

def execute_FCGF_Pipeline(voxel_size, inlier_th, subset, model, test_path, run_name):
    """Run the full FCGF pipeline: feature extraction, RANSAC, and ICP.

    Args:
        voxel_size (float): Downsampling size for point clouds.
        inlier_th (float): Inlier distance threshold.
        subset (int or None): Number of pairs to process; False for all.
        model (str): FCGF model path or identifier.
        test_path (str): Directory of test scenes.
        run_name (str): Base name for output folder.

    Returns:
        (output_folder: str, results: pd.DataFrame)
    """

    if subset:
       check_feasible_subset(subset)

    time = datetime.now(ZoneInfo("America/Sao_Paulo")).strftime('%Y-%m-%d_%H-%M-%S')
    output_folder = f"../output/FCGF/{run_name}-{time}"
    feature_path = f"{output_folder}/features"

    print("Applying FCGF to TEST split using:\n--> voxel_size=%f\n--> distance threshold=%f" %(voxel_size, inlier_th))
    print('======================================================')

    # For RANSAC, since it is called from a script, we don't need to use a Tee object just a simple StringIO
    ransac_tee_buffer = io.StringIO()

    # No redirect_stdout needed: run_benchmark streams directly to sys.__stdout__ and into the Tee buffer
    # Set Open3D's verbosity level to Debug to capture detailed iteration information
    o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Debug)
    # Execute FCGF with only the RANSAC stage
    run_benchmark(test_path, feature_path, voxel_size, inlier_th, subset, model, ransac_tee_buffer)
    # Returns Open3D's verbosity level to default mode
    o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)
    # Retrieve the captured output as a string
    ransac_captured_output = ransac_tee_buffer.getvalue()

    # Create thee Tee object to capture outputs from ICP stage while printing to the console
    icp_tee_buffer = Tee()

    # Redirect stdout to the Tee object during ICP execution
    with redirect_stdout(icp_tee_buffer):
        o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Debug)
        results = ICP_stage(output_folder, test_path, inlier_th, voxel_size)
        o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)
    icp_captured_output = icp_tee_buffer.getvalue()

    # Merge the iteration information into the main results DataFrame
    results = augment_results(results, ransac_captured_output, icp_captured_output)

    # Save the obtained table as a .csv in the output folder
    filename = f"{output_folder}/registration/registration_table_FCGF.csv"
    results.to_csv(filename, index=False)
    print('-------------------------------------------------------------------------------------------------------')
    print(f'Registration results table saved at: {filename}')
    

    # Retrieve times of preprocessing and ransac (handled by FCGF/scripts/benchmark_3dmatch.py) and summarize time results
    last_lines = ransac_captured_output.strip().splitlines()[-2:]       # break captured output into lines and
    for line in last_lines:                                             # look at only the last two lines, where time is printed
        line = line.strip().split()                                     # clean the line
        total_stage_times[line[0]] = float(line[-1].replace('s',''))    # {stage} total time = X.XXXXXs
    print('============================================== Time Summary ==============================================')
    stages_list = ['preprocessing', 'ransac', 'icp']
    for stage in stages_list:
        t = total_stage_times[stage]
        print(f"{stage} total time = {t:.5f}s")

    return output_folder, results

#### Explanation for Handling Output Differently in RANSAC and ICP

**1. Why We Can't Use `Tee()` for the RANSAC Case**  
We can't use the `Tee()` object during the RANSAC stage because RANSAC is executed via a separate Python script (`benchmark_3dmatch.py`) launched as a **subprocess**. The `Tee()` object works by overriding Python's `sys.stdout`, but **a subprocess does not share the same stdout environment** as the parent process. Thus, even if we redirected `sys.stdout` in the parent using `Tee()`, it would have **no effect** on the subprocess's output stream. Instead, we manually **pipe the subprocess's stdout back into Python**, read it line-by-line, and forward each line both to the actual console (`sys.__stdout__`) and to a `StringIO` buffer for later use. This allows us to simulate the behavior of `Tee` manually for this special case.

**2. Adapatations to `run_benchmark()`**  
The changes made were necessary to support this dual-streaming behavior. Specifically:
- We modified it to launch the subprocess with `stdout=subprocess.PIPE` and `text=True`, allowing us to read its output as lines of text.
- We read from `proc.stdout` line-by-line in real time to avoid buffering delays and forward each line immediately to both the console (`sys.__stdout__.write(line)`) and an optional buffer (`stdout.write(line)`).
- This approach ensures that subprocess output is both visible to the user and captured for later analysis (e.g., to extract iteration-level details).

These changes were crucial because we needed both **real-time feedback during execution** (e.g., for monitoring progress or debugging) and **post-hoc access to the raw logs** for analysis and DataFrame augmentation.

**3. Why We Can Use `Tee()` Normally for the ICP Case**  
In contrast, the ICP stage is **executed directly within the same Python process**, not via a subprocess. Therefore, `sys.stdout` redirection using a `Tee()` object works as expected. By using `with redirect_stdout(icp_tee_buffer):`, we effectively capture everything printed to the console during the ICP execution, while still displaying it live on the screen. Since Open3D outputs to standard Python stdout, this redirection allows us to seamlessly **capture and display** verbose debugging output during the ICP registration.

>**In short:**
>- RANSAC uses a subprocess → needs manual piping and forwarding.
>- ICP runs in-process → standard `Tee` redirection works perfectly.


---

# 6 Testing

Here we run the benchmark script to test the FCGF performance in both extracting the features itself and providing a registration based on these features. To check the whole implementation of this method as well as details on how this benchmark is performed, please refer to: [github.com/gabriel-corteletti/FCGF](https://github.com/gabriel-corteletti/FCGF).

In [27]:
subset = 1
voxel_size = 0.025
inlier_th = 0.05            # 5 cm --> this must be the same for ICP
model = fcgf_weight_path

run_name = "temp_testing_iteration_counter"

output_folder, results = execute_FCGF_Pipeline(voxel_size, inlier_th, subset, model, test_path, run_name)
display(results)

Applying FCGF to TEST split using:
--> voxel_size=0.025000
--> distance threshold=0.050000
06/05 18:35:54 ['../data/FCGF/threedmatch_test/7-scenes-redkitchen', '../data/FCGF/threedmatch_test/7-scenes-redkitchen-evaluation', '../data/FCGF/threedmatch_test/sun3d-home_at-home_at_scan1_2013_jan_1', '../data/FCGF/threedmatch_test/sun3d-home_at-home_at_scan1_2013_jan_1-evaluation', '../data/FCGF/threedmatch_test/sun3d-home_md-home_md_scan9_2012_sep_30', '../data/FCGF/threedmatch_test/sun3d-home_md-home_md_scan9_2012_sep_30-evaluation', '../data/FCGF/threedmatch_test/sun3d-hotel_uc-scan3', '../data/FCGF/threedmatch_test/sun3d-hotel_uc-scan3-evaluation', '../data/FCGF/threedmatch_test/sun3d-hotel_umd-maryland_hotel1', '../data/FCGF/threedmatch_test/sun3d-hotel_umd-maryland_hotel1-evaluation', '../data/FCGF/threedmatch_test/sun3d-hotel_umd-maryland_hotel3', '../data/FCGF/threedmatch_test/sun3d-hotel_umd-maryland_hotel3-evaluation', '../data/FCGF/threedmatch_test/sun3d-mit_76_studyroom-76-1study

,Scene,Target,Source,RANSAC Iterations,ICP Iterations,RANSAC: Fitness,ICP: Fitness,RANSAC: Inlier RMSE,ICP: Inlier RMSE,Initial Guess,Transformation
0,sun3d-hotel_umd-maryland_hotel3,8,16,26,7,0.391628,0.392747,0.013334,0.011841,"[[0.940662113516, -0.20566652082, 0.2699186366...","[[0.9351386543129522, -0.2176861532709953, 0.2..."
1,sun3d-home_md-home_md_scan9_2012_sep_30,9,11,34,14,0.354724,0.428941,0.019907,0.016680,"[[0.645964296982, 0.761812447988, -0.048702372...","[[0.6424222692739027, 0.766293676424384, 0.009..."
2,sun3d-hotel_umd-maryland_hotel1,11,13,42,30,0.742894,0.739206,0.015851,0.013639,"[[0.992939438593, -0.076018164794, 0.091063219...","[[0.9894401732833387, -0.08633699197725728, 0...."
3,7-scenes-redkitchen,36,38,61,22,0.640556,0.629767,0.019291,0.012374,"[[0.865750831936, 0.442241239846, -0.234303612...","[[0.8584262687155458, 0.43882503196751604, -0...."
4,sun3d-mit_76_studyroom-76-1studyroom2,36,49,46,30,0.499675,0.521213,0.018022,0.015734,"[[0.585597902115, -0.437327816728, 0.682509690...","[[0.574176984854038, -0.43621356419088947, 0.6..."


First we have the **feature extraction** stage, in which the metrics plotted are:
- Average Time.
- FPS (Features Per Second).
- time / feat.: Time necessary to extract one feature.

Then, we have the **feature evaluation** stage, in which the variables plotted are:
- $\tau_1$ **(Feature Distance Threshold)**  
    - Defines the maximum acceptable distance in feature space between the descriptors of a corresponding source/target pair for the putative feature match to be considered an **inlier (true positive)**.  
    - Ensures that feature descriptors are **sufficiently similar**.

- $\tau_2$ **(Inlier Recall Threshold)**  
    - Defines the **minimum acceptable inlier ratio** of a cloud pair (i.e., the percentage of keypoints that are correctly aligned under a **separate geometric distance threshold**).  
    - Ensures whether an alignment is **considered valid**.

- **Feature Match Recall (FMR) Computation**  
    - For each scene, and for the global average across all scenes: $ x.xxxx \pm y.yyyy $ (mean FMR and standard deviation).  
    - FMR computes the fraction of cloud pairs that achieve an **inlier ratio greater than** $ \tau_2 $ out of all possible pairs obtained.
    - The ground-truth transformation is used to **evaluate only the quality of the extracted features**, without considering the final registration step.

To determine whether a feature match and an alignment are successful, the evaluation follows these steps:

1. **Feature Matching:**  
    - Correspondences are found by performing a nearest neighbor search in **feature space**.
    - A match $(x_i, y_i)$ is considered valid if:
    $$ \| f(x_i) - f(y_i) \| \leq \tau_1 $$
    - This ensures that the descriptors are **sufficiently similar**.

2. **Ground-Truth Transformation Application:**  
    - The known ground-truth transformation $ T $ is applied to the source keypoints:
    $$ Tx_i $$

3. **Geometric Inlier Check:**  
    - A match is considered an **inlier** if the transformed source keypoint falls within a **fixed spatial distance threshold** $d_{\text{thresh}}$ (e.g., 10 cm) from the corresponding target keypoint:
    $$ \| Tx_i - y_i \| \leq d_{\text{thresh}} $$
    - This threshold is separate from \( \tau_1 \) and ensures **geometric correctness**.

4. **Inlier Ratio Computation:**  
    - The inlier ratio is computed as:
    $$ \text{Inlier Ratio} = \frac{\text{Number of inliers}}{\text{Total number of feature matches}} $$

5. **Alignment Success Criterion (τ₂):**  
    - If the inlier ratio is **greater than or equal to**  \( \tau_2 \)(e.g., 5%), the alignment is considered **successful**.

After the feature extraction and evaluation, the pairs are matched using **feature-based RANSAC**, a variation of standard RANSAC that incorporates **both geometric and feature similarity information**. The process consists of:
- Obtaining a set of **corresponding points** by performing a **nearest neighbor search in feature space** (matching each source keypoint to the target keypoint with the most similar feature descriptor).
- Running **standard RANSAC** on these correspondences to estimate a rigid transformation while filtering out outliers.

Finally, for **registration evaluation**, the algorithm attempts to align **all non-consecutive point clouds**. The algorithm determines whether a pair was successfully aligned by checking whether the **overlap** between the transformed source cloud and the target cloud exceeds a certain threshold (**30%**). If the alignment is deemed successful, the corresponding transformation is saved to a **log file**.


## 6.1 Quantitative Analysis

Now, we have one `.log` file for each scene. To evaluate the registration performance in terms of **Recall** and **Preccision**, we must use a MATLAB script provided by the author and adapted for our use case (the adapted script is available at: [github.com/gabriel-corteletti/3dmatch-toolbox](github.com/gabriel-corteletti/3dmatch-toolbox)).

Therefore, since we cannot run a MATLAB script in a Python environment, we must open MATLAB and run it from there using the `.log` files we obtained here (stored at `output/FCGF/{run_name}-{time}/registration/logs`)

However, if we are considering just a subset of the test split, we have to adapt also the ground truth files to consider only the results of the same alignments we performed, otherwise the computation of registration recall and precision will be affected by this inconsistency.

To do so, given the subset size we are considering, we first compute all possible pairs to be aligned, i.e. all non-consecutive pairs, and then we create a copy of the original ground truth *.log* and *.info* files where we select only the data related to these possible pairs.

We need to compute all the possible pairs instead of simply considering the same pairs inserted in the obtained registration .log files because it might happen that the algorithm judges itself not able to perform a certain alignment.

In [16]:
def get_matching_pairs(file):
    """
    Parse a matching-pairs text file and return a mapping of scene names to lists of matching ID pairs.

    Args:
        file (str): Path to the matching_pairs.txt file. Each block starts with a line `Set: <scene_name>`
                    followed by lines of `<tgt_ID> <src_ID>` pairs.

    Returns:
        dict[str, list[list[int]]]: A dictionary where keys are scene names and values are lists of
                                    [target_ID, source_ID] pairs.
    """

    # Initialize scene variable and dictionary that will store the collected info
    scene = None
    matching_pairs = defaultdict(list)
    
    # Read the matching_pairs.txt file
    with open(file, 'r') as f:
        # Iterate through each line
        for line in f:
            
            # Clean and split the line
            line = line.strip().split()

            # If the first word is 'Set:', it indicates the start of a block for a different scene
            if line[0] == 'Set:':
                scene = line[1]     # Update the current scene
                continue

            # If it is not a new scene, we are still in the same block
            elif scene:             # Check whether we truly are in a block
                # Collect the pairs for that scene
                matching_pairs[scene].append([int(line[0]), int(line[1])])
 
    return matching_pairs


def get_scene_reducedGT(log_path, info_path, scene_out_gt_path, matching_pairs, scene):
    """
    Filter and write reduced ground-truth files (gt.log and gt.info) for a single scene,
    including only specified matching pairs.

    Args:
        log_path (str): Path to the original gt.log file for this scene.
        info_path (str): Path to the original gt.info file for this scene.
        scene_out_gt_path (str): Directory where reduced files will be written.
        matching_pairs (dict[str, list[list[int]]]): Mapping of scene names to matching pairs.
        scene (str): Name of the scene to process (must be a key in matching_pairs).

    Returns:
        None
    """

    # Ensure output directory exists
    os.makedirs(scene_out_gt_path, exist_ok=True)

    # Output file paths
    out_log_path = os.path.join(scene_out_gt_path, 'gt.log')
    out_info_path = os.path.join(scene_out_gt_path, 'gt.info')

    # Compute number of fragments from pair count: N = (1 + sqrt(1+8*P)) / 2
    n_pairs = len(matching_pairs[scene])
    num_frag = int((1 + np.sqrt(1 + 8*n_pairs))/2)

    # Auxiliary flag to determine whether to copy the info of a pair
    copy = False

    # Open the complete gt log file in reading mode  
    with open(log_path, 'r') as i:
        # And the reduced gt log file in writing mode
        with open(out_log_path, 'w') as o:
            # Iterate through the lines of the complete file
            for idx, line in enumerate(i):

                # Clean and split the line
                line = line.strip().split()

                # Every 5 lines, we have a different pair info
                if idx%5 == 0:
                    # If the pair is included in the subset of matching pairs to be considered
                    if [int(line[0]), int(line[1])] in matching_pairs[scene]:
                        # Write <target_ID> <source_ID> <num_frag> and set flag to copy its block
                        o.write(f"{line[0]}\t {line[1]}\t {num_frag}\n")
                        copy = True
                    # Otherwise, reset copy flag
                    else:
                        copy = False
                # If it's not a header line and the flag is set, copy the info (it's a row of the matrix)
                elif copy:
                    o.write(f"{line[0]}\t {line[1]}\t {line[2]}\t {line[3]}\n")

    # Do the same procedure for the gt info file
    with open(info_path, 'r') as i:
        with open(out_info_path, 'w') as o:
            for idx, line in enumerate(i):
                line = line.strip().split()
                if idx%7 == 0:
                    if [int(line[0]), int(line[1])] in matching_pairs[scene]:
                        o.write(f"{line[0]}\t {line[1]}\t {num_frag}\n")
                        copy = True
                    else:
                        copy = False
                elif copy:
                    o.write(f"{line[0]}\t {line[1]}\t {line[2]}\t {line[3]}\t {line[4]}\t {line[5]}\n")


def get_reducedGT(output_folder, test_path):
    """
    Generate reduced ground-truth files for all scenes in test_path, based on matching pairs.

    This function reads the matching_pairs.txt in the registration folder of output_folder,
    then for each scene directory ending with '-evaluation' under test_path, it filters the
    gt.log and gt.info files to only include the pairs listed and writes them under
    output_folder/groundtruth/<scene>.

    Args:
        output_folder (str): Base output directory containing 'registration/matching_pairs.txt'.
        test_path (str): Directory containing scene subdirectories with '-evaluation' suffix.

    Returns:
        None
    """

    # Create groundtruth output directory
    out_gt_path = f"{output_folder}/groundtruth"
    os.makedirs(out_gt_path, exist_ok=True)

    # Obtain matching pairs to be considered
    matching_pairs = get_matching_pairs(f"{output_folder}/registration/matching_pairs.txt")

    # Iterate over scene directories
    for filename in os.listdir(test_path):
        
        # Check if the file name ends with 'evaluation'
        aux = filename.split('-')
        if aux[-1] == 'evaluation':

            # Store the path to the complete files in the evaluation folder
            log_path = os.path.join(test_path, filename, 'gt.log')
            info_path = os.path.join(test_path, filename, 'gt.info')

            # Define the path for the reduced groundtruth files
            scene_out_gt_path = os.path.join(out_gt_path, filename)
            
            # Obtain the scene name
            scene = '-'.join(aux[:-1])

            # Generate the reduuced files
            get_scene_reducedGT(log_path, info_path, scene_out_gt_path, matching_pairs, scene)
            
    print(f'Reduced ground truth files saved at: {out_gt_path}')

In [17]:
get_reducedGT(output_folder, test_path)

Reduced ground truth files saved at: ../output/FCGF/temp_testing_iteration_counter-2025-05-09_15-43-03/groundtruth


Then we run the script [evaluate.m](https://github.com/gabriel-corteletti/3dmatch-toolbox/blob/master/evaluation/geometric-registration/evaluate.m) to obtain a .csv file containing the registration recall and precision of each scene as well as the overall average results.

After obtaining this, we can upload it back to this environment and present it in a table below.

In [18]:
eval_csv_path = f"{output_folder}/evaluation/registration_evaluation_FCGF.csv"
if os.path.isfile(eval_csv_path):
    reg_eval_FCGF = pd.read_csv(eval_csv_path)
    display(reg_eval_FCGF)
else:
    print(f'No file found at: {eval_csv_path}\nPlease insert the .csv file with the obtained evaluation results in the expected path with the expected file name')

No file found at: ../output/FCGF/temp_testing_iteration_counter-2025-05-09_15-43-03/evaluation/registration_evaluation_FCGF.csv
Please insert the .csv file with the obtained evaluation results in the expected path with the expected file name


Besides computing **Recall** and **Precision** using the MATLAB script, we can also evaluate our previously computed performance metrics (`fitness` and `inlier RMSE`). To visualize the results **per scene** (instead of per alignment), we summarize as follows:

- **Per-scene statistics**:  
  Compute the mean and standard deviation of each metric (iteration counts, fitness, inlier RMSE) for each scene.

- **Overall statistics**:  
  Compute the mean and standard deviation of each metric across **all** alignments.

- **Inter-scene variation**:  
  Compute the standard deviation of the **per-scene means** for each metric, i.e. how much scene-to-scene averages vary.


In [19]:
def assess_results(results):
    """
    Given a registration result table, computes the mean performance (and standard deviation) of the
    alignment for all clouds of a specific scene and for that whole split we selected from a dataset.

    Args:
        results (pd.DataFrame): Table containing the registration results of the split

    Returns:
        pd.DataFrame: Table with the overall (mean and standard) performance results
    """

    # 1) Compute per-scene mean
    #    (Dropping 'Source','Target','Transformation' from the grouping)
    analysis_mean = results.copy()                                                                              # create a copy of results
    analysis_mean = analysis_mean.drop(["Source", "Target", "Initial Guess", "Transformation"], axis="columns")  # remove unnecessary columns
    analysis_mean = analysis_mean.groupby("Scene").mean().reset_index()                                         #  compute the averages of each scene
    analysis_mean = analysis_mean.rename(columns={"RANSAC: Fitness": "RANSAC: Mean Fitness",
                                                  "ICP: Fitness": "ICP: Mean Fitness",
                                                  "RANSAC: Inlier RMSE": "RANSAC: Mean Inlier RMSE",
                                                  "ICP: Inlier RMSE": "ICP: Mean Inlier RMSE",
                                                  "RANSAC Iterations": "Mean RANSAC Iterations",
                                                  "ICP Iterations": "Mean ICP Iterations"})

    # 2) Compute per-scene standard deviation
    analysis_std = results.copy()
    analysis_std = analysis_std.drop(["Source", "Target", "Initial Guess", "Transformation"], axis="columns")
    analysis_std = analysis_std.groupby("Scene").std().reset_index()
    analysis_std = analysis_std.rename(columns={"RANSAC: Fitness": "RANSAC: STD Fitness",
                                                "ICP: Fitness": "ICP: STD Fitness",
                                                "RANSAC: Inlier RMSE": "RANSAC: STD Inlier RMSE",
                                                "ICP: Inlier RMSE": "ICP: STD Inlier RMSE",
                                                "RANSAC Iterations": "STD RANSAC Iterations",
                                                "ICP Iterations": "STD ICP Iterations"})

    # 3) Merge the means and std columns side by side
    #    (We use 'Scene' as the key to match rows)
    analysis = pd.merge(analysis_mean, analysis_std, on="Scene", how="left")

    # 4) Compute overall means (using per-scene means) and overall std (using all samples) 
    total = pd.DataFrame({"Scene": "TOTAL",
                          "Mean RANSAC Iterations": analysis_mean["Mean RANSAC Iterations"].mean(),
                          "STD RANSAC Iterations": [results["RANSAC Iterations"].std()],
                          "Mean ICP Iterations": analysis_mean["Mean ICP Iterations"].mean(),
                          "STD ICP Iterations": [results["ICP Iterations"].std()],
                          "RANSAC: Mean Fitness": analysis_mean["RANSAC: Mean Fitness"].mean(),
                          "RANSAC: STD Fitness": [results["RANSAC: Fitness"].std()],
                          "ICP: Mean Fitness": analysis_mean["ICP: Mean Fitness"].mean(),
                          "ICP: STD Fitness": [results["ICP: Fitness"].std()],
                          "RANSAC: Mean Inlier RMSE": analysis_mean["RANSAC: Mean Inlier RMSE"].mean(),
                          "RANSAC: STD Inlier RMSE": [results["RANSAC: Inlier RMSE"].std()],
                          "ICP: Mean Inlier RMSE": analysis_mean["ICP: Mean Inlier RMSE"].mean(),
                          "ICP: STD Inlier RMSE": [results["ICP: Inlier RMSE"].std()]}, index=[0])

    # 5) Compute the inter_scene std deviation using the per-scene means
    inter_scenes_std = pd.DataFrame({"Scene": "Inter-Scene STD",
                                     "RANSAC: STD Fitness": [analysis["RANSAC: Mean Fitness"].std()],
                                     "ICP: STD Fitness": [analysis["ICP: Mean Fitness"].std()],
                                     "RANSAC: STD Inlier RMSE": [analysis["RANSAC: Mean Inlier RMSE"].std()],
                                     "ICP: STD Inlier RMSE": [analysis["ICP: Mean Inlier RMSE"].std()],
                                     "STD RANSAC Iterations": [analysis["Mean RANSAC Iterations"].std()],
                                     "STD ICP Iterations": [analysis["Mean ICP Iterations"].std()],
                                     # Leave the "mean" columns of the inter-scene std row blank
                                     "RANSAC: Mean Fitness": "",
                                     "ICP: Mean Fitness": "",
                                     "RANSAC: Mean Inlier RMSE": "",
                                     "ICP: Mean Inlier RMSE": "",
                                     "Mean RANSAC Iterations": "",
                                     "Mean ICP Iterations": ""}, index=[0])

    # Concatenate everything
    analysis = pd.concat([analysis, total, inter_scenes_std], ignore_index=True)

    # Reorder columns
    desired_order = ["Scene", "Mean RANSAC Iterations", "STD RANSAC Iterations", "Mean ICP Iterations", "STD ICP Iterations",
                     "RANSAC: Mean Fitness", "RANSAC: STD Fitness", "ICP: Mean Fitness", "ICP: STD Fitness",
                     "RANSAC: Mean Inlier RMSE", "RANSAC: STD Inlier RMSE", "ICP: Mean Inlier RMSE", "ICP: STD Inlier RMSE"]
    analysis = analysis[desired_order]

    # Ensures the evaluation folder is present
    os.makedirs(f"{output_folder}/evaluation", exist_ok=True)
    
    # Save the obtained registration results summary table as a .csv in the output folder
    filename = f"{output_folder}/evaluation/mean_fitness_and_RMSE_table_FCGF.csv"
    analysis.to_csv(filename, index=False)
    print(f'Registration results summary table saved at: {filename}')

    return analysis

In [20]:
analysis = assess_results(results)
display(analysis)

Registration results summary table saved at: ../output/FCGF/temp_testing_iteration_counter-2025-05-09_15-43-03/evaluation/mean_fitness_and_RMSE_table_FCGF.csv


,Scene,Mean RANSAC Iterations,STD RANSAC Iterations,Mean ICP Iterations,STD ICP Iterations,RANSAC: Mean Fitness,RANSAC: STD Fitness,ICP: Mean Fitness,ICP: STD Fitness,RANSAC: Mean Inlier RMSE,RANSAC: STD Inlier RMSE,ICP: Mean Inlier RMSE,ICP: STD Inlier RMSE
0,7-scenes-redkitchen,46.8,8.467585,13.6,2.408319,0.688231,0.134097,0.687854,0.131502,0.016621,0.002095,0.013846,0.002095
1,sun3d-home_at-home_at_scan1_2013_jan_1,85.2,132.599774,17.0,9.300538,0.740548,0.096650,0.741081,0.095503,0.016858,0.003666,0.013142,0.004010
2,sun3d-home_md-home_md_scan9_2012_sep_30,41.4,13.427584,17.8,8.105554,0.457737,0.321321,0.479877,0.320220,0.01978,0.005418,0.015561,0.005042
3,sun3d-hotel_uc-scan3,64.8,24.386472,19.4,7.231874,0.556031,0.126022,0.558481,0.125348,0.016837,0.001884,0.01367,0.001573
4,sun3d-hotel_umd-maryland_hotel1,47.833333,20.351085,16.666667,5.573748,0.635716,0.171652,0.639732,0.170476,0.01727,0.003309,0.014827,0.002872
5,sun3d-hotel_umd-maryland_hotel3,31.333333,10.503968,17.0,8.717798,0.574117,0.318136,0.565713,0.311608,0.016274,0.004546,0.011577,0.002909
6,sun3d-mit_76_studyroom-76-1studyroom2,42.25,20.500000,18.0,8.124038,0.636253,0.186772,0.638653,0.192343,0.014685,0.001531,0.012646,0.001773
7,sun3d-mit_lab_hj-lab_hj_tea_nov_2_2012_scan1_e...,71.0,56.506637,20.333333,8.386497,0.807367,0.008925,0.803857,0.007886,0.016248,0.000804,0.013474,0.000219
8,TOTAL,53.827083,51.643067,17.475,6.785781,0.637,0.199872,0.639406,0.196340,0.016822,0.003260,0.013593,0.002955
9,Inter-Scene STD,,18.044821,,2.013427,,0.110207,,0.104989,,0.001424,,0.001233


## 6.2 Qualitative Analysis

We can also perform a qualitative analysis through the visualization of a certain pair with the transformation we obtained. To do so, we use the same visualization function as always.

In [21]:
def draw_registration_result(source, target, transformation, voxel_size=0.0):
    """
    Plots the pair of target (cyan) and transformed source (yellow)
    cloud on top of each other. It also performs downsampling, if
    desired, to speed up the visualization.

    Args:
        source (open3d.geometry.PointCloud): Source cloud
        target (open3d.geometry.PointCloud): Target cloud
        transformation (numpy.ndarray): Transformation to be visualized
        voxel_size (float): Resulting size of voxels after downsampling,
                            if desired (default is no downsampling)

    Returns:
        open3d.geometry.PointCloud: Downsampled cloud
        open3d.registration.Feature: Features for registration
    """

    #create copies of both clouds to protect original data
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)

    #downsample for faster visualization (voxel_size is in meters)
    if (voxel_size):
        source_temp = source_temp.voxel_down_sample(voxel_size)
        target_temp = target_temp.voxel_down_sample(voxel_size)

    #paint target cyan and source yellow
    source_temp.paint_uniform_color([1, 0.706, 0])
    target_temp.paint_uniform_color([0, 0.651, 0.929])

    #apply the transformation to the source cloud
    source_temp.transform(transformation)

    #plot target and transformed source clouds
    o3d.visualization.draw_plotly([source_temp, target_temp], width=1200, height=800)

And also some auxiliary functions to retrieve the necessary information and clouds to plot the 3D graph.

In [22]:
def get_transformation(log_path, tgt_ID, src_ID):

    found = -1
    transformation = []

    with open(log_path, 'r') as f:
        for idx, line in enumerate(f):
            line = line.strip().split()
            if (idx%5 == 0) and (int(line[0]) == tgt_ID) and (int(line[1]) == src_ID):
                found = 0
            elif (found > -1) and (found < 4):
                transformation.append([float(i) for i in line])
                found += 1
                if found == 4:
                    break

    if transformation == []:
        transformation = None
    else:
        transformation = np.array(transformation)

    return transformation



def get_clouds(test_path, scene, tgt_ID, src_ID):
    
    target = source = False

    for filename in os.listdir(test_path):
        if filename == scene:

            frag_path = os.path.join(test_path, scene)
            for cloud in os.listdir(frag_path):
                cloud_ID = int(cloud.split('_')[-1].split('.')[0])
                if cloud_ID == src_ID:
                    cloud_path = os.path.join(frag_path, 'cloud_bin_%s.ply' %cloud_ID)
                    source = o3d.io.read_point_cloud(cloud_path)
                elif cloud_ID == tgt_ID:
                    cloud_path = os.path.join(frag_path, 'cloud_bin_%s.ply' %cloud_ID)
                    target = o3d.io.read_point_cloud(cloud_path)
    return target, source



def plot_alignment(output_folder, test_path, inlier_th, scene, tgt_ID, src_ID, voxel_size=0.0, alignment='ICP'):

    if alignment not in {'NONE', 'RANSAC', 'ICP'}:
        raise ValueError('alignment must be one of:\n -> NONE: initial relative pose\n'
                                                    ' -> RANSAC: registration obtained after RANSAC\n'
                                                    ' -> ICP: final registration obtained after ICP\n'
                        f'[Got {alignment} instead]')

    target, source = get_clouds(test_path, scene, tgt_ID, src_ID)
    
    if not target or not source:
        print("ERROR: clouds not found\nCheck if the selected clouds' IDs are part of the selected scene")
        return

    print(f"> Scene: {scene}")
    print(f"> Target: {tgt_ID}")
    print(f"> Source: {src_ID}")
    print("------------------------")

    if alignment == 'NONE':
        transformation = np.eye(4)
        print('INITIAL RELATIVE POSE (BEFORE REGISTRATION)')

    elif alignment == 'RANSAC':
        guess_log_path = f'{output_folder}/registration/initial_guesses_logs/{scene}_FCGF.log'
        transformation = get_transformation(guess_log_path, tgt_ID, src_ID)
        print('RANSAC REGISTRATION (INITIAL GUESS)')

    elif alignment == "ICP":
        log_path = f"{output_folder}/registration/logs/{scene}_FCGF.log"
        transformation = get_transformation(log_path, tgt_ID, src_ID)
        print('ICP REGISTRATION (FINAL REGISTRATION)')

    print("------------------------")
    if transformation is None:
            print('This pair could not be aligned')
            return

    eval = o3d.pipelines.registration.evaluate_registration(source, target, inlier_th, transformation)
    
    print(f"> Fitness: {eval.fitness}")
    print(f"> RMSE: {eval.inlier_rmse}")
    if alignment != 'NONE':
        print(f"> Obtained Transformation:\n{transformation}")
    draw_registration_result(source, target, transformation, voxel_size)

Then you can invoke it by specifying:

- **scene**: the scene name  
- **target** and **source** fragment IDs  
- **voxel_size**: used here solely for downsampling  
- **alignment_type**: one of `"NONE"` (initial pose), `"RANSAC"`, or `"ICP"`

In [23]:
scenes = ['7-scenes-redkitchen',                                # index: 0
          'sun3d-home_at-home_at_scan1_2013_jan_1',             # index: 1
          'sun3d-home_md-home_md_scan9_2012_sep_30',            # index: 2
          'sun3d-hotel_uc-scan3',                               # index: 3
          'sun3d-hotel_umd-maryland_hotel1',                    # index: 4
          'sun3d-hotel_umd-maryland_hotel3',                    # index: 5
          'sun3d-mit_76_studyroom-76-1studyroom2',              # index: 6
          'sun3d-mit_lab_hj-lab_hj_tea_nov_2_2012_scan1_erika'] # index: 7

plot_alignment(output_folder, test_path, inlier_th,
               scene=scenes[5],
               tgt_ID=10, src_ID=14, voxel_size=0.025, alignment='ICP')

> Scene: sun3d-hotel_umd-maryland_hotel3
> Target: 10
> Source: 14
------------------------
ICP REGISTRATION (FINAL REGISTRATION)
------------------------
This pair could not be aligned


---